# Ladybug and RelateDB through QuackWeave

Within each benchmark profile, Ladybug and RelateDB use the same QuackWeave build for DuckDB 2.0.1, SQL text, and LSQB message–creator pairs. Each provider runs in a fresh process for each block. The timed interval covers execution and fetching all Arrow rows. The sections through “Full-scale query failures” report the original baseline recorded in the three `mod*.json` files beside this notebook. The final section compares that baseline with the optimized full-scale run in `optimized/mod1.json`.

The queries below run in DuckDB and aggregate those saved measurements. The graph queries themselves ran in the pinned benchmark host. Refresh the SQL cells with this notebook’s directory as the working directory.

## What happened at full scale

Both files contained the same 3,809,607 creator edges. A bare GQL lookup returned one author in a median 2.81 ms through Ladybug and 6.94 s through RelateDB. Returning that author’s 6,465 messages took 5.20 ms and 6.90 s respectively. DuckDB’s watchlist join and full aggregate completed through Ladybug in 358.40 ms and 96.60 ms. RelateDB returned `QW_EXECUTION` on both, and on the `GRAPH_TABLE` equality filter.

The Ladybug watchlist plan read every edge before the SQL join. The Ladybug aggregate delivered one row across the provider boundary. At 119,052 edges, the same aggregate through RelateDB delivered every matching edge to DuckDB. Peak process RSS at full scale was 701 MiB for Ladybug and 4,064 MiB for RelateDB; the latter measures its successful standalone workloads.

In [9]:
SELECT oracle.edges AS edges, oracle.people AS people, oracle.fanout AS rows_for_selected_author,
       blocks AS blocks_per_provider, runs AS timed_repeats_per_block,
       round(fixture.files."social.lbdb".bytes / 1048576.0, 1) AS ladybug_mib,
       round(fixture.files."social.rl".bytes / 1048576.0, 1) AS relate_mib
FROM read_json_auto('mod*.json', union_by_name=true)
ORDER BY edges

edges,people,rows_for_selected_author,blocks_per_provider,timed_repeats_per_block,ladybug_mib,relate_mib
14883,6960,22,4,4,4.4,4.3
119052,10223,190,4,4,11.0,29.0
3809607,10947,6465,2,3,189.8,149.1


## Query latency

The table shows wall-clock and process CPU medians. Each worker first runs a point lookup and one untimed pass through the workloads. Subsequent iterations shuffle the workload order. Each pair of blocks reverses provider order. P95 is nearest rank and is sensitive to the small sample count.

In [10]:
WITH samples AS (
    SELECT r.oracle.edges AS edges, w.provider AS provider, s."case" AS workload,
           s.wall_ms AS wall_ms, s.cpu_ms AS cpu_ms
    FROM read_json_auto('mod*.json', union_by_name=true) AS r,
         unnest(r.workers) AS blocks(w), unnest(w.samples) AS observations(s)
)
SELECT edges, workload, provider, count(*) AS observations,
       round(median(wall_ms), 2) AS p50_ms, round(quantile_disc(wall_ms, .95), 2) AS p95_ms,
       round(median(cpu_ms), 2) AS median_cpu_ms
FROM samples GROUP BY ALL ORDER BY edges, workload, provider

edges,workload,provider,observations,p50_ms,p95_ms,median_cpu_ms
14883,cypher_fanout,ladybug,16,0.64,0.9,0.64
14883,cypher_fanout,relate,16,31.97,42.68,75.75
14883,full_scan_aggregate,ladybug,16,5.16,5.77,28.11
14883,full_scan_aggregate,relate,16,45.32,53.16,89.2
14883,gql_fanout,ladybug,16,2.84,3.31,2.84
14883,gql_fanout,relate,16,34.06,40.85,77.08
14883,gql_lookup,ladybug,16,2.23,2.73,2.24
14883,gql_lookup,relate,16,34.02,41.18,77.84
14883,graph_table_where,ladybug,16,0.67,0.88,0.67
14883,graph_table_where,relate,16,46.75,52.52,90.18


In [11]:
WITH samples AS (
    SELECT r.oracle.edges AS edges, w.provider AS provider, s."case" AS workload,
           s.wall_ms AS wall_ms, s.cpu_ms AS cpu_ms
    FROM read_json_auto('mod*.json', union_by_name=true) AS r,
         unnest(r.workers) AS blocks(w), unnest(w.samples) AS observations(s)
)
SELECT edges, workload,
       round(median(wall_ms) FILTER (WHERE provider='relate') /
             median(wall_ms) FILTER (WHERE provider='ladybug'), 2) AS latency_ratio
FROM samples GROUP BY edges, workload ORDER BY edges, workload

edges,workload,latency_ratio
14883,cypher_fanout,49.93
14883,full_scan_aggregate,8.78
14883,gql_fanout,12.0
14883,gql_lookup,15.23
14883,graph_table_where,69.95
14883,table_watchlist,6.44
119052,cypher_fanout,255.12
119052,full_scan_aggregate,25.99
119052,gql_fanout,64.74
119052,gql_lookup,81.3


## Graph rows delivered to DuckDB

These are actual operator row counts from separate `EXPLAIN ANALYZE` runs. They distinguish the number of graph rows delivered to DuckDB from the number of rows returned to the client. Each watchlist is an ordinary DuckDB temporary table containing the same five keys.

In [12]:
SELECT r.oracle.edges AS edges, x.workload, x.ladybug_rows, x.relate_rows
FROM read_json_auto('mod*.json', union_by_name=true) AS r, LATERAL (VALUES
  ('graph_table_where', r.parity.ladybug.provider_rows.graph_table_where, r.parity.relate.provider_rows.graph_table_where),
  ('table_watchlist', r.parity.ladybug.provider_rows.table_watchlist, r.parity.relate.provider_rows.table_watchlist),
  ('full_scan_aggregate', r.parity.ladybug.provider_rows.full_scan_aggregate, r.parity.relate.provider_rows.full_scan_aggregate)
) AS x(workload, ladybug_rows, relate_rows)
ORDER BY edges, workload

edges,workload,ladybug_rows,relate_rows
14883,full_scan_aggregate,1,14883
14883,graph_table_where,22,14883
14883,table_watchlist,14883,14883
119052,full_scan_aggregate,1,119052
119052,graph_table_where,190,119052
119052,table_watchlist,119052,119052
3809607,full_scan_aggregate,1,NULL
3809607,graph_table_where,6465,NULL
3809607,table_watchlist,3809607,NULL


## Registration, first lookup and memory

Registration includes read-only graph admission and automatic catalog discovery. The first lookup uses a fresh DuckDB connection, with OS file caches left intact. RSS is the peak for the whole worker process, including DuckDB, the provider and Arrow.

In [13]:
SELECT r.oracle.edges AS edges, w.provider, count(*) AS processes,
       round(median(w.setup.register_ms), 2) AS registration_ms,
       round(median(w.first_lookup.wall_ms), 2) AS first_lookup_ms,
       round(max(w.worker_peak_rss_mib), 1) AS max_worker_rss_mib
FROM read_json_auto('mod*.json', union_by_name=true) AS r, unnest(r.workers) AS blocks(w)
GROUP BY r.oracle.edges, w.provider ORDER BY edges, w.provider

edges,provider,processes,registration_ms,first_lookup_ms,max_worker_rss_mib
14883,ladybug,4,33.83,22.64,188.6
14883,relate,4,40.46,39.36,156.2
119052,ladybug,4,33.96,22.17,205.0
119052,relate,4,229.04,197.84,411.6
3809607,ladybug,2,76.82,21.81,701.0
3809607,relate,2,8490.51,7208.0,4063.9


## Full-scale query failures

A missing latency or graph-scan row count represents a query that did not complete. With the full SF1 projection, QuackWeave returned `QW_EXECUTION` on the RelateDB hybrid equality filter, table watchlist, and aggregate. A direct read-only probe of the underlying two-column graph scan reported an expression capacity limit of 268,435,456 bytes (256 MiB). The probe is diagnostic and does not contribute a timing sample.

In [14]:
WITH statuses AS (
    SELECT r.oracle.edges AS edges, s."case" AS workload,
           CASE WHEN s.ladybug.n = 0 THEN 'failed' ELSE 'passed' END AS ladybug_status,
           CASE WHEN s.relate.n = 0 THEN 'failed' ELSE 'passed' END AS relate_status
    FROM read_json_auto('mod*.json', union_by_name=true) AS r, unnest(r.summary) AS rows(s)
)
SELECT * FROM statuses WHERE edges = (SELECT max(edges) FROM statuses) ORDER BY workload

edges,workload,ladybug_status,relate_status
3809607,cypher_fanout,passed,passed
3809607,full_scan_aggregate,passed,failed
3809607,gql_fanout,passed,passed
3809607,gql_lookup,passed,passed
3809607,graph_table_where,passed,failed
3809607,table_watchlist,passed,failed


## Full-scale results after optimization

Compare the six workloads on all 3,809,607 message–creator pairs. Median and p95 wall times include fetching the result. The speedup divides the baseline median by the optimized median, so values above 1 indicate faster execution. A blank baseline time with a positive optimized sample count identifies a workload that now completes. Sample counts are the successful timed executions for each provider.

In [15]:
WITH baseline AS (
    SELECT unnest(summary) AS s FROM read_json_auto('mod1.json')
), optimized AS (
    SELECT unnest(summary) AS s FROM read_json_auto('optimized/mod1.json')
)
SELECT b.s."case" AS workload, p.provider,
       p.baseline.n AS baseline_n, p.optimized.n AS optimized_n,
       round(p.baseline.p50_ms, 2) AS baseline_p50_ms,
       round(p.optimized.p50_ms, 2) AS optimized_p50_ms,
       round(p.optimized.p95_ms, 2) AS optimized_p95_ms,
       round(p.baseline.p50_ms / nullif(p.optimized.p50_ms, 0), 2) AS speedup
FROM baseline AS b
JOIN optimized AS o ON o.s."case" = b.s."case"
CROSS JOIN LATERAL (VALUES
    ('Ladybug', b.s.ladybug, o.s.ladybug),
    ('RelateDB', b.s.relate, o.s.relate)
) AS p(provider, baseline, optimized)
ORDER BY workload, provider

workload,provider,baseline_n,optimized_n,baseline_p50_ms,optimized_p50_ms,optimized_p95_ms,speedup
cypher_fanout,Ladybug,6,6,2.62,5.79,6.77,0.45
cypher_fanout,RelateDB,6,6,6979.12,8.82,10.71,791.69
full_scan_aggregate,Ladybug,6,6,96.6,180.34,292.61,0.54
full_scan_aggregate,RelateDB,0,6,NULL,4522.34,5147.26,NULL
gql_fanout,Ladybug,6,6,5.2,11.66,13.21,0.45
gql_fanout,RelateDB,6,6,6895.71,12.69,15.61,543.39
gql_lookup,Ladybug,6,6,2.81,6.07,8.86,0.46
gql_lookup,RelateDB,6,6,6937.4,4.09,5.01,1697.46
graph_table_where,Ladybug,6,6,3.5,7.57,8.19,0.46
graph_table_where,RelateDB,0,6,NULL,10.24,12.57,NULL


### Ladybug controls on the full-scale graph

These controls rerun both extension builds on the same Ladybug graph. The mixed run exercises all six workloads on each connection. The isolated run gives each workload its own connection. Compare the old and new p50 wall times within each run. A positive percentage means the new build took longer.

In [16]:
WITH controls AS (
    SELECT j.key AS workload, ends_with(r.filename, 'ladybug-isolated.json') AS isolated,
           (j.value->'baseline'->>'p50_ms')::DOUBLE AS old_ms,
           (j.value->'optimized'->>'p50_ms')::DOUBLE AS new_ms
    FROM read_json_objects(['optimized/ladybug-paired.json', 'optimized/ladybug-isolated.json'],
                           filename = true) AS r, json_each(r.json, '$.summary') AS j
)
SELECT m.workload,
       round(m.old_ms, 2) AS mixed_old_ms, round(m.new_ms, 2) AS mixed_new_ms,
       round(100 * (m.new_ms / nullif(m.old_ms, 0) - 1), 1) AS mixed_change_pct,
       round(i.old_ms, 2) AS isolated_old_ms, round(i.new_ms, 2) AS isolated_new_ms,
       round(100 * (i.new_ms / nullif(i.old_ms, 0) - 1), 1) AS isolated_change_pct
FROM controls AS m JOIN controls AS i USING (workload)
WHERE NOT m.isolated AND i.isolated
ORDER BY workload

workload,mixed_old_ms,mixed_new_ms,mixed_change_pct,isolated_old_ms,isolated_new_ms,isolated_change_pct
cypher_fanout,4.53,5.75,26.9,6.38,5.05,-20.9
full_scan_aggregate,142.6,172.73,21.1,170.34,186.45,9.5
gql_fanout,9.25,11.05,19.5,9.08,10.37,14.2
gql_lookup,4.75,5.56,17.0,3.31,3.58,8.3
graph_table_where,6.01,7.43,23.7,4.23,4.44,5.1
table_watchlist,593.48,16.65,-97.2,597.92,11.74,-98.0
